In [2]:
import numpy as np
import scipy.stats as stats
from fitter import Fitter

In [ ]:
np.random.seed(1)
 
# Q3 DATA
q3_demand = np.array([
    2723463, 2464345, 2223094, 2738725, 3199565,
    3212175, 2356641, 1680760,  994350,  872841,
    2146563, 2647838, 1511062, 1895864, 2418814,
    1011569, 1126667, 2551808, 3246210
], dtype=float)
 
spot_prices = np.array([
    21.14, 21.70, 17.73, 12.75, 26.97, 16.72, 20.22, 13.83, 17.22, 19.87,
    17.21, 26.12, 15.42, 19.67, 19.16, 12.78, 16.54, 21.68, 23.75, 17.59,
    19.00, 17.39, 17.36, 23.63, 20.05, 16.80, 25.51, 17.44, 25.84, 18.61,
    12.71, 16.18, 13.46, 21.76, 26.68, 22.28, 17.57, 13.51, 21.12, 26.89,
    26.70, 23.93, 24.91, 14.97, 24.69, 25.12, 24.50, 17.31, 15.42, 17.38,
    18.42, 21.19, 15.89, 19.52, 12.54, 14.05, 23.82, 18.10, 25.02, 16.60,
    24.92, 16.47, 25.22, 25.97, 23.31, 21.99, 23.95, 18.27, 14.36, 14.69,
    18.61, 14.16, 17.49, 21.27, 14.20
], dtype=float)
 
# A2 FIT DISTRIBUTIONS
f_demand = Fitter(q3_demand,   distributions=["norm"])
f_spot   = Fitter(spot_prices, distributions=["beta"])
f_demand.fit()
f_spot.fit()
 
demand_params = list(f_demand.get_best(method="sumsquare_error").values())[0]
spot_params   = list(f_spot.get_best(method="sumsquare_error").values())[0]
 
print("Distribution Parameters")
print(f"Demand (Normal): loc={demand_params['loc']:,.2f}, scale={demand_params['scale']:,.2f}")
print(f"Spot   (Beta)  : a={spot_params['a']:.4f}, b={spot_params['b']:.4f}, "
      f"loc={spot_params['loc']:.4f}, scale={spot_params['scale']:.4f}")
 
# SHARED CONSTANTS
SELL_PRICE    = 150.00
VARIABLE_COST = 102.50
SALVAGE       =   5.30
FIXED_COSTS   = 55_757_500.00   # SG&A $37,920,000 + D&R $17,837,500
 
def _draw(n):
    """Draw n samples of (demand, spot) from fitted distributions."""
    demands = np.maximum(stats.norm.rvs(**demand_params, size=n), 0)
    spots   = np.maximum(stats.beta.rvs(**spot_params,   size=n), 0)
    return demands, spots
 
def _ci90(arr):
    return stats.t.interval(0.90, len(arr)-1, loc=np.mean(arr), scale=stats.sem(arr))

#  PART A — LONG-TERM PURCHASE CONTRACT  (lcd_cost = $17.00)
 
# A1 PROFIT FUNCTION 
def purchase_profit(order_size, demand, spot_price):
    revenue   = demand * SELL_PRICE
    mfg_cost  = demand * VARIABLE_COST
    lcd_total = order_size * 17.00
    salvage   = max(order_size - demand, 0) * SALVAGE
    spot_cost = max(demand - order_size, 0) * spot_price
    return (revenue + salvage) - (mfg_cost + lcd_total + spot_cost + FIXED_COSTS)
 
def simulate_purchase(order_qty, n=10_000):
    demands, spots = _draw(n)
    return np.array([purchase_profit(order_qty, d, s) for d, s in zip(demands, spots)])
 
# A3 EXPECTED PROFIT  
profits_A3 = simulate_purchase(2_170_000)
print("\nA3: Expected Profit at 2,170,000 units")
print(f"Expected Profit: ${np.mean(profits_A3):,.2f}")
 
# A4 OPTIMAL ORDER QUANTITY
print("\nA4: Optimal Order Quantity Search")
order_range   = np.arange(1_000_000, 3_600_000, 100_000)
mean_profits  = {qty: np.mean(simulate_purchase(qty)) for qty in order_range}
best_qty_A4   = max(mean_profits, key=mean_profits.get)
profits_A4    = simulate_purchase(best_qty_A4)
ci_A4         = _ci90(profits_A4)
 
print(f"Optimal Order Quantity  : {best_qty_A4:,} units")
print(f"Optimal Expected Profit : ${np.mean(profits_A4):,.2f}")
print(f"90% Confidence Interval : [${ci_A4[0]:,.2f}, ${ci_A4[1]:,.2f}]")
 

#  PART B — OPTION CONTRACT  

EXERCISE_PRICE = 16.75
RESERVATION    =  0.50
 
# B1 — PROFIT FUNCTION 
def option_profit(order_size, option_qty, demand, spot_price):
    revenue     = demand    * SELL_PRICE
    mfg_cost    = demand    * VARIABLE_COST
    lcd_total   = order_size * 17.00
    reservation = option_qty * RESERVATION
    salvage     = max(order_size - demand, 0) * SALVAGE
    shortfall   = max(demand - order_size, 0)
 
    # Exercise option only if cheaper than spot
    if spot_price > EXERCISE_PRICE:
        opt_cost  = min(shortfall, option_qty) * EXERCISE_PRICE
        spot_cost = max(shortfall - option_qty, 0) * spot_price
    else:
        opt_cost  = 0
        spot_cost = shortfall * spot_price
 
    return (revenue + salvage) - (mfg_cost + lcd_total + reservation + opt_cost + spot_cost + FIXED_COSTS)
 
def simulate_option(order_qty, option_qty, n=10_000):
    demands, spots = _draw(n)
    return np.array([option_profit(order_qty, option_qty, d, s) for d, s in zip(demands, spots)])
 
#  B2 — OPTIMAL OPTION QUANTITY  
print("\nB2: Optimal Option Quantity (Order = 2,170,000 units)")
option_range  = np.arange(0, 3_600_000, 100_000)
opt_means_B2  = {oq: np.mean(simulate_option(2_170_000, oq)) for oq in option_range}
best_opt_B2   = max(opt_means_B2, key=opt_means_B2.get)
profits_B2    = simulate_option(2_170_000, best_opt_B2)
ci_B2         = _ci90(profits_B2)
 
print(f"Optimal Option Quantity : {best_opt_B2:,} units")
print(f"Optimal Expected Profit : ${np.mean(profits_B2):,.2f}")
print(f"90% Confidence Interval : [${ci_B2[0]:,.2f}, ${ci_B2[1]:,.2f}]")
 
#  B3 — COMPARE A3 vs B2 
print("\nB3: Comparison — Purchase (A3) vs Option (B2) at Order = 2,170,000 units")
print(f"{'Metric':<30} {'Purchase (A3)':>18} {'Option (B2)':>18}")
print("-" * 68)
print(f"{'Expected Profit':<30} ${np.mean(profits_A3):>17,.2f} ${np.mean(profits_B2):>17,.2f}")
print(f"{'Std Deviation (Risk)':<30} ${np.std(profits_A3):>17,.2f} ${np.std(profits_B2):>17,.2f}")
for p in [10, 25, 50, 75, 90]:
    print(f"{'P'+str(p)+' Percentile':<30} ${np.percentile(profits_A3, p):>17,.2f} ${np.percentile(profits_B2, p):>17,.2f}")
 
#  B4 — OPTIMAL OPTION QUANTITY  
print(f"\nB4: Optimal Option Quantity (Order = {best_qty_A4:,} units from A4)")
opt_means_B4  = {oq: np.mean(simulate_option(best_qty_A4, oq)) for oq in option_range}
best_opt_B4   = max(opt_means_B4, key=opt_means_B4.get)
profits_B4    = simulate_option(best_qty_A4, best_opt_B4)
ci_B4         = _ci90(profits_B4)
 
print(f"Optimal Option Quantity : {best_opt_B4:,} units")
print(f"Optimal Expected Profit : ${np.mean(profits_B4):,.2f}")
print(f"90% Confidence Interval : [${ci_B4[0]:,.2f}, ${ci_B4[1]:,.2f}]")
 
print(f"\nB4 vs A4 Comparison (Order = {best_qty_A4:,} units)")
print(f"{'Metric':<30} {'Purchase (A4)':>18} {'Option (B4)':>18}")
print("-" * 68)
print(f"{'Expected Profit':<30} ${np.mean(profits_A4):>17,.2f} ${np.mean(profits_B4):>17,.2f}")
print(f"{'Std Deviation (Risk)':<30} ${np.std(profits_A4):>17,.2f} ${np.std(profits_B4):>17,.2f}")
for p in [10, 25, 50, 75, 90]:
    print(f"{'P'+str(p)+' Percentile':<30} ${np.percentile(profits_A4, p):>17,.2f} ${np.percentile(profits_B4, p):>17,.2f}")

Distribution Parameters
Demand (Normal): loc=2,159,071.26, scale=751,724.48
Spot   (Beta)  : a=0.9417, b=0.8548, loc=12.5394, scale=14.4306

A3: Expected Profit at 2,170,000 units
Expected Profit: $5,869,699.19

A4: Optimal Order Quantity Search
Optimal Order Quantity  : 1,200,000 units
Optimal Expected Profit : $7,065,763.56
90% Confidence Interval : [$6,701,315.03, $7,430,212.09]

B2: Optimal Option Quantity (Order = 2,170,000 units)
Optimal Option Quantity : 1,200,000 units
Optimal Expected Profit : $5,801,416.19
90% Confidence Interval : [$5,344,884.78, $6,257,947.60]

B3: Comparison — Purchase (A3) vs Option (B2) at Order = 2,170,000 units
Metric                              Purchase (A3)        Option (B2)
--------------------------------------------------------------------
Expected Profit                $     5,869,699.19 $     5,801,416.19
Std Deviation (Risk)           $    26,359,925.52 $    27,751,179.84
P10 Percentile                 $   -30,403,473.04 $   -31,866,360.60
P2

In [10]:
# PART A VERIFICATION
print("\nPart A: Demand=2,000,000 | Spot=$20")
print(f"Profit: ${purchase_profit(2_170_000, 2_000_000, 20):,.2f}")
 
# PART B VERIFICATION 
cases = [
    (2_170_000, 1_000_000, 3_000_000, 24, 35_450_000),
    (2_170_000, 1_000_000, 4_000_000, 24, 60_182_500),
    (2_170_000, 1_000_000, 4_000_000, 12, 74_892_500),
]
 
print("\nPart B:")
print(f"{'Case':<6} {'Demand':>12} {'Option Q':>10} {'Spot':>6} {'Profit':>18} {'Expected':>18} {'Match':>6}")
for i, (oq, optq, d, s, expected) in enumerate(cases, 1):
    result = option_profit(oq, optq, d, s)
    match  = "PASS" if abs(result - expected) < 1 else "FAIL"
    print(f"{i:<5} {d:>12,} {optq:>10,} {s:>6} ${result:>17,.2f} ${expected:>17,} {match:>6}")
 


Part A: Demand=2,000,000 | Spot=$20
Profit: $3,253,500.00

Part B:
Case         Demand   Option Q   Spot             Profit           Expected  Match
1        3,000,000  1,000,000     24 $    35,450,000.00 $       35,450,000   PASS
2        4,000,000  1,000,000     24 $    60,182,500.00 $       60,182,500   PASS
3        4,000,000  1,000,000     12 $    74,892,500.00 $       74,892,500   PASS
